In [1]:
import asyncio
import time
import aiohttp

In [2]:
async def say_after(delay, what):
    print("start say", what)
    await asyncio.to_thread(time.sleep, delay)
    print(what)



async def main():
    t1 = time.time()

    task1 = asyncio.create_task(say_after(1, "hello"))
    task2 = asyncio.create_task(say_after(2, "world"))

    await asyncio.gather(task1, task2)
    
    t2 = time.time()
    print("total time", t2 - t1)


await main()

start say hello
start say world
hello
world
total time 2.0064990520477295


In [5]:
def async_timeit(coro):
    async def inner(*args, **kwargs):
        t1 = time.time()
        try:
            return await coro(*args, **kwargs)
        finally:
            t2 = time.time()
            print(f"time of `{coro.__name__}`", t2 - t1)
    return inner

In [6]:
async def say_after(delay, what):
    print("start say", what)
    await asyncio.to_thread(time.sleep, delay)
    print(what)


@async_timeit
async def main():
    task1 = asyncio.create_task(say_after(1, "hello"))
    task2 = asyncio.create_task(say_after(2, "world"))

    await asyncio.gather(task1, task2)


# main = async_timeit(main)


await main()

start say hello
start say world
hello
world
time of `main` 2.005929946899414


In [24]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
URLS = [URL] * 20


@async_timeit
async def fetch_url(url, session):
    async with session.get(url) as resp:
        text = await resp.text()
        return (resp.status, len(text))


async def fetch_batch_urls(urls):
    async with aiohttp.ClientSession() as session:
        tasks = []
        for url in urls:
            tasks.append(fetch_url(url, session))
    
        resp = await asyncio.gather(*tasks)
        print(resp)


@async_timeit
async def run():
    await fetch_batch_urls(URLS)


await run()

time of `fetch_url` 1.0047061443328857
time of `fetch_url` 1.2383060455322266
time of `fetch_url` 1.2455322742462158
time of `fetch_url` 1.2481520175933838
time of `fetch_url` 1.3054859638214111
time of `fetch_url` 1.3124988079071045
time of `fetch_url` 1.313628911972046
time of `fetch_url` 1.318040132522583
time of `fetch_url` 1.3196299076080322
time of `fetch_url` 1.3223190307617188
time of `fetch_url` 1.3228480815887451
time of `fetch_url` 1.394059181213379
time of `fetch_url` 1.402336835861206
time of `fetch_url` 1.4226679801940918
time of `fetch_url` 1.4647459983825684
time of `fetch_url` 1.4747319221496582
time of `fetch_url` 1.4763469696044922
time of `fetch_url` 1.5571868419647217
time of `fetch_url` 1.5597786903381348
time of `fetch_url` 1.5606341361999512
[(200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840)

In [26]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
URLS = [URL] * 20


@async_timeit
async def fetch_url(url, session, sem):
    async with sem:
        async with session.get(url) as resp:
            text = await resp.text()
            return (resp.status, len(text))


async def fetch_batch_urls(urls, sem):
    async with aiohttp.ClientSession() as session:
        tasks = []
        for url in urls:
            tasks.append(fetch_url(url, session, sem))
    
        resp = await asyncio.gather(*tasks)
        print(resp)


@async_timeit
async def run():
    sem = asyncio.Semaphore(2)
    await fetch_batch_urls(URLS, sem)


await run()

time of `fetch_url` 0.3662891387939453
time of `fetch_url` 0.45990896224975586
time of `fetch_url` 0.48465776443481445
time of `fetch_url` 0.5451083183288574
time of `fetch_url` 0.599045991897583
time of `fetch_url` 0.6393637657165527
time of `fetch_url` 0.7061288356781006
time of `fetch_url` 0.7228250503540039
time of `fetch_url` 0.8106467723846436
time of `fetch_url` 0.8218088150024414
time of `fetch_url` 0.9152052402496338
time of `fetch_url` 0.9413669109344482
time of `fetch_url` 1.0616860389709473
time of `fetch_url` 1.094040870666504
time of `fetch_url` 1.1842937469482422
time of `fetch_url` 1.2239251136779785
time of `fetch_url` 1.2889752388000488
time of `fetch_url` 1.3517138957977295
time of `fetch_url` 1.3955929279327393
time of `fetch_url` 1.4895069599151611
[(200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554

In [30]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
URLS = [URL] * 20
WORKERS = 4
BATCH = len(URLS) // WORKERS


async def fetch_url(url, session):
    async with session.get(url) as resp:
        text = await resp.text()
        return (resp.status, len(text))


@async_timeit
async def fetch_batch_urls(urls, session):
    resps = []
    for url in urls:
        resps.append(
            await asyncio.create_task(fetch_url(url, session))
        )

    return resps


@async_timeit
async def run():
    async with aiohttp.ClientSession() as session:
        tasks = [
            fetch_batch_urls(URLS[i * BATCH : (i + 1) * BATCH], session)
            for i in range(WORKERS)
        ]
        res = await asyncio.gather(*tasks)
        print(res)


await run()

time of `fetch_batch_urls` 0.89951491355896
time of `fetch_batch_urls` 0.9343008995056152
time of `fetch_batch_urls` 1.0452392101287842
time of `fetch_batch_urls` 1.1715121269226074
[[(200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840)], [(200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840)], [(200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840)], [(200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840)]]
time of `run` 1.1733644008636475


In [35]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
URLS = [URL] * 20
WORKERS = 5
BATCH = len(URLS) // WORKERS


async def fetch_url(url, session):
    async with session.get(url) as resp:
        text = await resp.text()
        return (resp.status, len(text))


@async_timeit
async def fetch_batch_worker(que, session):
    resps = []
    cnt = 0
    while True:
        url = await que.get()
        if url is None:
            await que.put(url)
            break

        resps.append(await asyncio.create_task(fetch_url(url, session)))
        cnt += 1

    print("num of urls", cnt)

    return resps


@async_timeit
async def run():
    que = asyncio.Queue()
    for url in URLS:
        await que.put(url)
    await que.put(None)
        
    async with aiohttp.ClientSession() as session:
        workers = [
            fetch_batch_worker(que, session)
            for i in range(WORKERS)
        ]
        res = await asyncio.gather(*workers)
        print(res)


await run()

num of urls 3
time of `fetch_batch_worker` 1.0963659286499023
num of urls 5
time of `fetch_batch_worker` 1.1640419960021973
num of urls 5
time of `fetch_batch_worker` 1.2676467895507812
num of urls 5
time of `fetch_batch_worker` 1.2808952331542969
num of urls 2
time of `fetch_batch_worker` 1.282485008239746
[[(200, 554840), (200, 554840)], [(200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840)], [(200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840)], [(200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840)], [(200, 554840), (200, 554840), (200, 554840)]]
time of `run` 1.284668207168579


In [36]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
URLS = [URL] * 20
WORKERS = 5
BATCH = len(URLS) // WORKERS


async def fetch_url(url, session):
    async with session.get(url) as resp:
        text = await resp.text()
        return (resp.status, len(text))


@async_timeit
async def fetch_batch_worker(que, session):
    resps = []
    cnt = 0
    while True:
        url = await que.get()
        resps.append(await asyncio.create_task(fetch_url(url, session)))
        cnt += 1
        que.task_done()

    print("num of urls", cnt)

    return resps


@async_timeit
async def run():
    que = asyncio.Queue()
    for url in URLS:
        await que.put(url)
        
    async with aiohttp.ClientSession() as session:
        workers = [
            asyncio.create_task(fetch_batch_worker(que, session))
            for i in range(WORKERS)
        ]

        await que.join()

        for worker in workers:
            worker.cancel()

await run()

time of `fetch_batch_worker` 0.8773598670959473
time of `fetch_batch_worker` 0.8775310516357422
time of `fetch_batch_worker` 0.8775520324707031
time of `fetch_batch_worker` 0.8775720596313477
time of `fetch_batch_worker` 0.8775937557220459
time of `run` 0.8798210620880127


## generator

In [38]:
async def counter(n):
    for i in range(n):
        yield i
        print(f"yield {i=}")
        await asyncio.sleep(0.1)


async for x in counter(5):
    print(x)

0
yield i=0
1
yield i=1
2
yield i=2
3
yield i=3
4
yield i=4


In [40]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
URLS = [URL] * 20


@async_timeit
async def fetch_url(url, session):
    async with session.get(url) as resp:
        text = await resp.text()
        return (resp.status, len(text))


async def fetch_batch_urls(urls):
    async with aiohttp.ClientSession() as session:
        tasks = []
        for url in urls:
            tasks.append(fetch_url(url, session))

        start = time.time()
        async for compl_task in asyncio.as_completed(tasks):
            print("compl_task", time.time() - start)

            res_task = await compl_task
            print("res_task", time.time() - start)


@async_timeit
async def run():
    await fetch_batch_urls(URLS)


await run()

time of `fetch_url` 0.8398890495300293
compl_task 0.8418700695037842
res_task 0.8419058322906494
time of `fetch_url` 1.1368892192840576
time of `fetch_url` 1.137342929840088
compl_task 1.1386909484863281
res_task 1.138718843460083
compl_task 1.1387948989868164
res_task 1.1388180255889893
time of `fetch_url` 1.1566739082336426
time of `fetch_url` 1.15681791305542
compl_task 1.158653736114502
res_task 1.1586987972259521
time of `fetch_url` 1.1580591201782227
time of `fetch_url` 1.1588108539581299
compl_task 1.160881757736206
res_task 1.1609108448028564
compl_task 1.1615290641784668
res_task 1.1615610122680664
compl_task 1.161585807800293
res_task 1.161604642868042
time of `fetch_url` 1.1642851829528809
time of `fetch_url` 1.165112018585205
compl_task 1.166165828704834
res_task 1.1661899089813232
compl_task 1.1662147045135498
res_task 1.166233777999878
time of `fetch_url` 1.2220079898834229
time of `fetch_url` 1.222593069076538
compl_task 1.2239158153533936
res_task 1.2239549160003662
com

In [41]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
URLS = [URL] * 20


@async_timeit
async def fetch_url(url, session, name):
    async with session.get(url) as resp:
        text = await resp.text()
        return (resp.status, len(text), name)


async def fetch_batch_urls(urls):
    async with aiohttp.ClientSession() as session:
        tasks = []
        for i, url in enumerate(urls):
            name = f"fetch_{i}"
            tasks.append(fetch_url(url, session, name))

        async for compl_task in asyncio.as_completed(tasks):
            res = await compl_task

            idx = int(res[2][6:])
            if idx % 2 == 0:
                yield res


@async_timeit
async def run():
    async for x in fetch_batch_urls(URLS):
        print("result:", x)


await run()

time of `fetch_url` 1.152082920074463
result: (200, 554840, 'fetch_2')
time of `fetch_url` 1.3765568733215332
result: (200, 554840, 'fetch_10')
time of `fetch_url` 1.3904860019683838
time of `fetch_url` 1.4034879207611084
time of `fetch_url` 1.4349329471588135
time of `fetch_url` 1.4372210502624512
time of `fetch_url` 1.4373109340667725
time of `fetch_url` 1.4380993843078613
time of `fetch_url` 1.4396495819091797
result: (200, 554840, 'fetch_16')
result: (200, 554840, 'fetch_4')
result: (200, 554840, 'fetch_18')
result: (200, 554840, 'fetch_14')
time of `fetch_url` 1.4433820247650146
time of `fetch_url` 1.4432241916656494
result: (200, 554840, 'fetch_8')
time of `fetch_url` 1.5069561004638672
time of `fetch_url` 1.5100018978118896
time of `fetch_url` 1.5214200019836426
time of `fetch_url` 1.575331211090088
result: (200, 554840, 'fetch_6')
time of `fetch_url` 1.580470085144043
time of `fetch_url` 1.6347029209136963
result: (200, 554840, 'fetch_0')
time of `fetch_url` 1.6756701469421387


In [44]:
2 ^ 3 == 1

True

In [45]:
3 ^ 4 == 1

False

In [47]:
4 ^ 3 == 1

False

In [48]:
n ^ (n + 1) == 1

NameError: name 'n' is not defined

In [ ]:
1000 1